# 🩺 Medical RAG Assistant
### Retrieval-Augmented Generation on Medical PDFs — Interview Ready

---

**What this notebook does:**
Upload any medical PDF → ask natural language questions → get answers grounded in the document with page citations.

**The RAG idea in one sentence:**
> Instead of asking the LLM from memory, we *retrieve* the relevant text from the document first, then ask the LLM to answer *only from that text*.

**Tech Stack:**
| Step | Tool |
|---|---|
| PDF Loading | `langchain-community` PyPDFLoader |
| Text Chunking | `langchain` RecursiveCharacterTextSplitter |
| Embeddings | `sentence-transformers` — all-MiniLM-L6-v2 (free, CPU) |
| Vector Database | `chromadb` (local, no server needed) |
| LLM | Google Gemini 2.5 Flash via `langchain-google-genai` |
| RAG Chain | `langchain` RetrievalQA |

✅ Runs fully in **Google Colab** — no local setup needed.

---
## Step 1 — Install Dependencies

Run this once at the start of your Colab session.  
These are the only packages this project needs.

In [1]:
!pip install -q \
langchain \
langchain-community \
langchain-groq \
langchain-text-splitters \
langchain-classic \
chromadb \
sentence-transformers \
pypdf \
datasets \
ragas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.4/245.4 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 12.9 MB/s eta 0:00:00


---
## Step 2 — Imports

All imports in one place — easy to see every tool being used.

In [2]:
import os

# PDF loading
from langchain_community.document_loaders import PyPDFLoader

# Text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings — local model, no API key needed
from langchain_community.embeddings import HuggingFaceEmbeddings

# Vector database
from langchain_community.vectorstores import Chroma

# LLM —
from langchain_groq import ChatGroq

# RAG chain — ties retriever + LLM together
from langchain_classic.chains import RetrievalQA

print("✅ Imports successful")

/tmp/ipykernel_12078/3015226578.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ Imports successful


---
## Step 3 — Upload Your Medical PDF

Run this cell → a file picker will appear → choose your PDF.  
Works directly in Google Colab — no file paths to configure.

In [3]:
from google.colab import files

print("📁 Select your medical PDF file...")
uploaded = files.upload()

# Get the filename of the uploaded file
pdf_filename = list(uploaded.keys())[0]

print(f"\n✅ Uploaded: {pdf_filename}")
print(f"   Size: {len(uploaded[pdf_filename]) / 1024:.1f} KB")

📁 Select your medical PDF file...


Saving Comprehensive Medical Knowledge Base.pdf to Comprehensive Medical Knowledge Base (2).pdf

✅ Uploaded: Comprehensive Medical Knowledge Base (2).pdf
   Size: 62.5 KB


---
## Step 4 — Load the PDF

**PyPDFLoader** reads the PDF and gives us one `Document` object per page.  
Each `Document` contains the page text + metadata (filename, page number).

In [4]:
loader = PyPDFLoader(pdf_filename)
pages = loader.load()

print(f"✅ PDF loaded successfully")
print(f"   Pages found: {len(pages)}")
print(f"\n--- Preview of Page 1 ---")
print(pages[0].page_content[:500])

✅ PDF loaded successfully
   Pages found: 16

--- Preview of Page 1 ---
Comprehensive Medical Knowledge Base for RAG
Testing
Introduction to Human Health
Human health is influenced by genetics, nutrition, environment, exercise, sleep, mental well-being, and
healthcare access. The human body contains multiple interconnected systems including the cardiovascular
system, respiratory system, digestive system, endocrine system, nervous system, immune system, and
musculoskeletal system.
A healthy adult typically maintains normal blood pressure between 90/60 mmHg and 120/80


---
## Step 5 — Split Text into Chunks

**Why do we chunk?**  
LLMs have a limited context window — we can't send the entire PDF at once.  
We split the document into small, overlapping chunks. When a question comes in,  
we only send the *most relevant* chunks to the LLM.

- `chunk_size = 1000` → each chunk is ~1000 characters  
- `chunk_overlap = 200` → chunks overlap by 200 characters so context isn't lost at boundaries

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = splitter.split_documents(pages)

print(f"✅ Document split into {len(chunks)} chunks")
print(f"\n--- Sample Chunk ---")
print(f"From page: {chunks[0].metadata.get('page', 0) + 1}")
print(f"Text: {chunks[0].page_content[:300]}...")

✅ Document split into 17 chunks

--- Sample Chunk ---
From page: 1
Text: Comprehensive Medical Knowledge Base for RAG
Testing
Introduction to Human Health
Human health is influenced by genetics, nutrition, environment, exercise, sleep, mental well-being, and
healthcare access. The human body contains multiple interconnected systems including the cardiovascular
system, re...


---
## Step 6 — Load the Embedding Model

**What is an embedding?**  
An embedding converts text into a list of numbers (a vector) that captures *meaning*.  
Sentences with similar meaning get similar vectors.

**Model: `all-MiniLM-L6-v2`**
- Produces 384-dimensional vectors
- Free, runs on CPU, no API key needed
- Downloads automatically (~90 MB, cached after first run)

In [6]:
print("⏳ Loading embedding model (downloads ~90MB on first run)...")

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

# Quick test — embed one sentence
test_vector = embedding_model.embed_query("What are the symptoms?")

print(f"✅ Embedding model loaded")
print(f"   Vector dimensions: {len(test_vector)}")

⏳ Loading embedding model (downloads ~90MB on first run)...


/tmp/ipykernel_12078/2325195177.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded
   Vector dimensions: 384


---
## Step 7 — Create the Vector Database (ChromaDB)

**What happens here?**  
Every chunk gets converted into an embedding vector and stored in ChromaDB.  
This is the "index" — we only do this once.

**At query time:**  
The user's question is also embedded → ChromaDB finds the chunks whose vectors are  
closest to the question vector → those are the most relevant chunks.

In [7]:
print(f"⏳ Embedding {len(chunks)} chunks and storing in ChromaDB...")

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
)

print(f"✅ Vector database created")
print(f"   Total vectors stored: {vectordb._collection.count()}")

⏳ Embedding 17 chunks and storing in ChromaDB...
✅ Vector database created
   Total vectors stored: 17


---
## Step 8 — Create the Retriever

The retriever is the search interface over the vector database.  
Given any query, it returns the top-K most relevant chunks.

`k=3` → fetch the 3 most relevant chunks for each question.

In [8]:
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

# Test the retriever with a sample question
test_question = "What are the main topics covered in this document?"
retrieved_docs = retriever.invoke(test_question)

print(f"✅ Retriever ready — fetches top 3 chunks per question")
print(f"\n🔍 Test retrieval for: '{test_question}'")
for i, doc in enumerate(retrieved_docs, 1):
    page = doc.metadata.get("page", 0) + 1
    print(f"\n  Chunk {i} (Page {page}):")
    print(f"  {doc.page_content[:200]}...")

✅ Retriever ready — fetches top 3 chunks per question

🔍 Test retrieval for: 'What are the main topics covered in this document?'

  Chunk 1 (Page 16):
  Public Health
Public health focuses on disease prevention and community health improvement.
Public Health Measures
Vaccination programs
Clean water supply
Sanitation
Health education
Conclusion
Health...

  Chunk 2 (Page 12):
  Antibiotics are ineffective against viral infections.
Analgesics
Analgesics reduce pain.
Examples:
Paracetamol
Ibuprofen
Overuse of NSAIDs may damage the stomach lining.
Medical Imaging
MRI
Magnetic R...

  Chunk 3 (Page 4):
  Chest pain
Risk Groups
Elderly
Infants
Immunocompromised individuals
Treatment
Antibiotics
Antiviral medications
Oxygen therapy
Hydration
Endocrine Disorders
Diabetes Mellitus
Diabetes mellitus is a c...


---
## Step 9 — Add Your Gemini API Key

Get a free key at: **https://aistudio.google.com/app/apikey**

Paste it below when prompted.

In [9]:
import os
import getpass

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Groq API Key: ")

print("✅ API key saved")

Enter Groq API Key: ··········
✅ API key saved


---
## Step 10 — Initialize the Gemini LLM

**Why Gemini 2.5 Flash?**
- Fast response time
- Large context window (fits multiple retrieved chunks easily)
- Free tier available

**Why `temperature=0`?**  
For medical QA, we want factual and deterministic answers — not creative ones.  
Temperature 0 means the model always picks the most likely (safest) answer.

In [10]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=os.environ["GROQ_API_KEY"],

    model_name="llama-3.1-8b-instant",

    temperature=0.1
)

print("✅ Groq model initialized")

✅ Groq model initialized


---
## Step 11 — Build the RetrievalQA Chain

**This is where RAG comes together.**

`RetrievalQA` is a LangChain chain that does everything automatically:
1. Takes your question
2. Retrieves the top-K relevant chunks from ChromaDB
3. Injects those chunks as context into the prompt
4. Sends it to Gemini
5. Returns the grounded answer

`chain_type="stuff"` = stuffs all retrieved chunks directly into the prompt (simplest approach, works well with small k).

`return_source_documents=True` = also returns which chunks were used, so we can show citations.

In [11]:

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)
print("✅ RAG chain ready")
print("   Pipeline: Question → Retriever → ChromaDB → Gemini → Answer + Sources")

✅ RAG chain ready
   Pipeline: Question → Retriever → ChromaDB → Gemini → Answer + Sources


---
## Step 12 — Ask a Question!

Change the question below and run the cell.  
The answer is generated purely from your uploaded PDF — no hallucinations from outside knowledge.

In [12]:
question = "What are the main symptoms oh heart attack?"

# Run the RAG pipeline
result = qa_chain.invoke({"query": question})

# Print answer
print("\n💡 ANSWER:\n")

print(result["result"])


💡 ANSWER:

The main symptoms of a heart attack include:

1. Chest pain
2. Sweating
3. Nausea
4. Shortness of breath

It's also worth noting that some people may experience pain in their left arm, fatigue, or other symptoms, but the above four are the most commonly recognized signs of a heart attack.


---
## Step 13 — Ask Questions

Run this cell to ask multiple questions in a row.  
Type `quit` to stop.

In [13]:
question = input("❓ Enter your medical question: ")

# Run the RAG pipeline
result = qa_chain.invoke({"query": question})

# Print Question
print("\n" + "=" * 60)
print("❓ QUESTION")
print("=" * 60)
print(question)

# Print Answer
print("\n" + "=" * 60)
print("💡 ANSWER")
print("=" * 60)
print(result["result"])

# Print Retrieved Sources
print("\n" + "=" * 60)
print("📚 RETRIEVED SOURCES")
print("=" * 60)

for i, doc in enumerate(result["source_documents"][:2], 1):

    page = doc.metadata.get("page", 0) + 1

    print(f"\n📄 Source {i} — Page {page}")
    print("-" * 40)

    print(doc.page_content[:300] + "...")

❓ Enter your medical question: i am in depression how to come out of it

❓ QUESTION
i am in depression how to come out of it

💡 ANSWER
I'm so sorry to hear that you're struggling with depression. Coming out of depression can be a challenging and individualized process, but there are some general steps and strategies that may help. Please keep in mind that if you're experiencing severe symptoms or suicidal thoughts, it's essential to seek immediate help from a mental health professional or a crisis hotline.

Here are some steps to help you come out of depression:

1. **Seek professional help**: Consult a mental health professional, such as a therapist or counselor, who can help you develop a treatment plan and provide support. They can also help you identify underlying causes of your depression.
2. **Medication**: Your doctor may prescribe antidepressant medication to help alleviate symptoms. However, medication should be used in conjunction with therapy and lifestyle changes.
3. **Ther

****

In [24]:
from datasets import Dataset

test_questions = {
    "question": [
        "What are the symptoms of hypertension?",
        "What is tuberculosis?",
        "What are common imaging techniques?"
    ],

    "expected_keywords": [
        ["headache", "dizziness", "blurred vision", "fatigue"],

        ["bacterial", "lungs", "infection"],

        ["MRI", "CT", "X-ray", "Ultrasound"]
    ]
}

eval_dataset = Dataset.from_dict(test_questions)

print("✅ Evaluation dataset ready")

✅ Evaluation dataset ready


In [16]:
eval_dataset = eval_dataset.remove_columns(
    [col for col in eval_dataset.column_names if col in ["answer", "contexts"]]
)

eval_dataset = eval_dataset.add_column("answer", answers)

eval_dataset = eval_dataset.add_column("contexts", contexts)

print("✅ Dataset updated")

✅ Dataset updated


In [28]:
total_accuracy = 0

print("📊 RAG Evaluation Metrics")
print("=" * 70)

questions = test_questions["question"]
keywords_list = test_questions["expected_keywords"]

for i, (question, keywords) in enumerate(
    zip(questions, keywords_list), 1
):

    # Generate answer
    result = qa_chain.invoke({"query": question})

    answer = result["result"].lower()

    # Keyword match score
    matched = sum(
        1 for keyword in keywords
        if keyword.lower() in answer
    )

    accuracy = matched / len(keywords)

    total_accuracy += accuracy

    # Print results
    print(f"\n🧪 Test Case {i}")
    print("-" * 70)

    print(f"❓ Question: {question}")

    print(f"\n✅ Accuracy Score: {accuracy:.2f}")

# Overall score
overall_accuracy = total_accuracy / len(questions)

print("\n" + "=" * 70)
print("📈 OVERALL METRICS")
print("=" * 70)

print(f"✅ Overall Accuracy: {overall_accuracy:.2f}")

print("=" * 70)

📊 RAG Evaluation Metrics

🧪 Test Case 1
----------------------------------------------------------------------
❓ Question: What are the symptoms of hypertension?

✅ Accuracy Score: 1.00

🧪 Test Case 2
----------------------------------------------------------------------
❓ Question: What is tuberculosis?

✅ Accuracy Score: 1.00

🧪 Test Case 3
----------------------------------------------------------------------
❓ Question: What are common imaging techniques?

✅ Accuracy Score: 0.75

📈 OVERALL METRICS
✅ Overall Accuracy: 0.92
